# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [ ]:
!pip install mteb

In [ ]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [ ]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

In [ ]:
from mteb import EncoderProtocol
from mteb.similarity_functions import cos_sim

In [ ]:
from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"

In [ ]:
class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        if self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        if self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = torch.stack([
                    t.mean(dim=1).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ])
            if (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                embeddings = outputs.last_hidden_state.mean(dim=1)


        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy, overlap):
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        else:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(
                    outputs.last_hidden_state, numbers_of_chunks
                )
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped
                ])
            else:
                embeddings = self.__get_eos_token_embedding(
                    outputs.last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings

## LongEmbed LEMBWikimQARetrieval

In [ ]:
import mteb

In [ ]:
long_embed_wiki_task = mteb.get_task("LEMBWikimQARetrieval")

In [ ]:
import json
import gc

strategies = [
    {
        "name": "Chunking",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 0
    },
    {
        "name": "Chunking 64 Overlap",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 64
    },
    {
        "name": "First",
        "strategy": Strategy.first,
        "max_size": 512,
        "overlap": None
    },
]

main_scores_roberta = []
roberta = None

for strategy in strategies:
    roberta = XMLRoBERTa(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(roberta, encode_kwargs={"batch_size": 4})
    main_scores_roberta.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"roberta_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del roberta
    torch.cuda.empty_cache()
    gc.collect()

main_scores_qwen3 = []
qwen3 = None

for strategy in strategies:
    qwen3 = Qwen3_Embedding(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(qwen3, encode_kwargs={"batch_size": 4})
    main_scores_qwen3.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"qwen3_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del qwen3
    torch.cuda.empty_cache()
    gc.collect()

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


...

Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


Converting corpus dict:   0%|          | 0/300 [00:00<?, ? examples/s]

(300, 1024)


In [ ]:
main_scores_roberta

[0.01584, 0.01657, 0.01804]

In [ ]:
main_scores_qwen3

[0.76854, 0.76813, 0.56949]

In [ ]:
pd.DataFrame(
    {
        "Strategy": [s["name"] for s in strategies],
        "Qwen3 score": main_scores_qwen3,
        "XML-RoBERTa score": main_scores_roberta
    }
).set_index("Strategy")


,Qwen3 score,XML-RoBERTa score
Strategy,,
Chunking,0.76854,0.01584
Chunking 64 Overlap,0.76813,0.01657
First,0.56949,0.01804


# Memory module

In [ ]:
from torch import nn


class Memory(nn.Module):

    def __init__(self, hidden_state_dim):
        super().__init__()
        self.hidden_state_dim = hidden_state_dim
        self.gated_update = GatedUpdate(self.hidden_state_dim)
        self.attention = Attention(self.hidden_state_dim)
        self.memory_cells = []

    def reset(self):
        self.memory_cells = []

    def update_memory(self, input):
        if len(memory_cells) == 0:
            self.memory_cells.append(input)
            return input
        else:
            attention = self.attention(input)
            self.gated_update(input, self.memory_cells, attention)

class GatedUpdate(nn.Module):

    def __init__(self, hidden_state_dim):
        super().__init__()
        self.Wz = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Uz = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Wr = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Ur = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Wc = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Uc = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def forward(self, input, memory, attention):
        z = nn.functional.sigmoid(self.Wz(input) + self.Uz(memory))
        r = nn.functional.sigmoid(self.Wr(input) + self.Ur(memory))

        c_tilde = nn.functional.tanh(self.Wc(input) + self.Uc(r * memory))
        return (1 - z) * memory + z * c_tilde

class Attention(nn.Module):

    def __init__(self, hidden_state_dim):
        super().__init__()
        self.hidden_state_dim = hidden_state_dim
        self.Wk = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)
        self.Wq = nn.Linear(self.hidden_state_dim, self.hidden_state_dim)

    def forward(self, input):
        return nn.functional.softmax(
            self.Wk(input) @ self.Wq(input).T
        )